In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install -q transformers peft datasets accelerate

In [3]:
!pip install -q "peft==0.15.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 15.8 MB/s eta 0:00:00


In [4]:
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,DefaultDataCollator
)

from datasets import Dataset
import torch
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

In [5]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [6]:
#Question-1:

label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

train["label"] = train["answer"].map(label_map)

print("Answer:", train.loc[150, "answer"])
print("Encoded label:", train.loc[150, "label"])

Answer: C
Encoded label: 2


In [7]:
#Question-2:

row0 = train.iloc[0]

option_b_input = str(row0["prompt"]) + " [SEP] " + str(row0["B"])

print(option_b_input)
print("Character length:", len(option_b_input))

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Character length: 407


In [8]:
#Question-3:

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

row = train.iloc[0]

choices = []

for option in ["A", "B", "C", "D", "E"]:
    choices.append(
        str(row["prompt"]) + " [SEP] " + str(row[option])
    )

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].unsqueeze(0)

print(input_ids.shape)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

torch.Size([1, 5, 128])


In [9]:
#Question-4:

batch_size = 16
num_choices = 5
sequence_length = 128

print(batch_size * num_choices * sequence_length)

10240


In [10]:
#Question-5

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
mc_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

row = train.iloc[0]

choices = [
    str(row["prompt"]) + " [SEP] " + str(row[option])
    for option in ["A", "B", "C", "D", "E"]
]

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

inputs = {
    "input_ids": encoding["input_ids"].unsqueeze(0),
    "attention_mask": encoding["attention_mask"].unsqueeze(0)
}

outputs = mc_model(**inputs)

print(outputs.logits.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.Size([1, 5])


In [11]:
#Question-6

row = train.iloc[0]

choices = [
    str(row["prompt"]) + " [SEP] " + str(row[option])
    for option in ["A", "B", "C", "D", "E"]
]

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

inputs = {
    "input_ids": encoding["input_ids"].unsqueeze(0),
    "attention_mask": encoding["attention_mask"].unsqueeze(0),
    "labels": torch.tensor([row["label"]])
}

outputs = mc_model(**inputs)

print("Loss:", outputs.loss)
print("Loss shape:", outputs.loss.shape)
print("Number of dimensions:", outputs.loss.dim())

Loss: tensor(1.6055, grad_fn=<NllLossBackward0>)
Loss shape: torch.Size([])
Number of dimensions: 0


In [12]:
#Question-7

# Reload a fresh multiple-choice model
mc_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

# Apply LoRA
mc_model = get_peft_model(mc_model, lora_config)

# Count trainable parameters
trainable_params = sum(
    p.numel() for p in mc_model.parameters() if p.requires_grad
)

print("Trainable parameters:", trainable_params)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 295681


In [13]:
import transformers, peft, torch
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("torch:", torch.__version__)

transformers: 5.0.0
peft: 0.15.2
torch: 2.10.0+cpu


In [14]:
train_small = train.iloc[:32].copy()

dataset = Dataset.from_pandas(train_small)

In [15]:
def preprocess(example):
    choices = [
        str(example["prompt"]) + " [SEP] " + str(example[option])
        for option in ["A", "B", "C", "D", "E"]
    ]

    encoding = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=64,
    )

    encoding["labels"] = example["label"]
    return encoding

tokenized_dataset = dataset.map(preprocess)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

In [16]:
tokenized_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [17]:
training_args = TrainingArguments(
    output_dir="./mcq_lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to="none",
    remove_unused_columns=False
)

In [18]:
trainer = Trainer(
    model=mc_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DefaultDataCollator(),
)

In [19]:
#Question-9
train_result = trainer.train()

print("Global Step:", trainer.state.global_step)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,1.568453
2,1.660987
3,1.596474
4,1.610635


Global Step: 4


In [20]:
#Question-10

mc_model.eval()

device = next(mc_model.parameters()).device

row = train.iloc[0]

choices = [
    str(row["prompt"]) + " [SEP] " + str(row[option])
    for option in ["A", "B", "C", "D", "E"]
]

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].unsqueeze(0).to(device)
attention_mask = encoding["attention_mask"].unsqueeze(0).to(device)

with torch.no_grad():
    outputs = mc_model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

probs = torch.softmax(outputs.logits, dim=1)

print("Probabilities:", probs.cpu())
print("Option E probability:", probs[0, 4].item())

Probabilities: tensor([[0.2025, 0.2048, 0.2013, 0.1955, 0.1959]])
Option E probability: 0.1958935707807541
